# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4: CTR / Engagement Opportunity Scoring.** I'm starting with the CTR half and will add engagement later only if the session data is dense enough.

The question: among pages that already rank on the first two result pages and get real impressions, which ones get far fewer clicks than other pages at the same position, and so are worth a title, meta description or snippet review first?

Why this lane:

- The signal it needs (impressions, clicks, position) is the densest part of the data, so I'm not betting seven weeks on a sparse column.
- A page that ranks but doesn't get clicked has a cheap, concrete fix to try. Editing a title takes far less effort than rewriting a whole article.
- The opportunity is measured against an expectation (CTR for that position), not against a label someone defined by rule. That keeps me away from the `trend_direction` proxy label I saw in the Week 1 notebooks.

The setup cell below loads the starter data. Pages with `avg_position = 0` are dropped because, per the data dictionary, that means "no data", not rank zero.

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages from {df['client_id'].nunique()} clients, {df.shape[1]} columns")

# avg_position = 0 means "no position data", not rank zero, so those rows are left out
has_pos = df["avg_position"] > 0
print(f"Pages with no position data (excluded): {(~has_pos).sum():,}")

30,000 pages from 32 clients, 44 columns
Pages with no position data (excluded): 1,205


## 2. The question: decision, action, cost of a wrong call

**Decision it improves.** An SEO editor has time to review maybe 20 to 50 pages a week. Which pages should go to the top of that list?

**Unit of analysis.** One page (one `content_id`), using its trailing 90-day numbers. Only pages with an average position between 1 and 20 and at least 500 impressions in 90 days count as candidates. Below that, one or two clicks swing the CTR too much to mean anything.

**Output.** A ranked review queue. Each page gets:

- a gap score: how far its CTR sits below what pages at a similar position usually get, weighted by how many impressions it has;
- reason codes a person can check, such as high impressions, strong position, CTR under half the tier median, or zero clicks;
- a suggested action: rewrite title/meta, check whether the page matches the search intent, or just monitor.

**Who acts and how.** An editor opens the top pages and decides whether the title and snippet are actually weak. The editor makes the call; the queue only orders the reading.

**Cost of a wrong call.**

- A false positive (page flagged but its CTR is fine for its query mix): wasted review time, and a risk of rewriting a title that was working.
- A false negative (a weak page that never makes the list): clicks keep being lost on a page that already has the ranking.

With limited review time, false positives at the top of the list hurt most. So the main metric will be precision at the reviewer's capacity (precision@20 / @50), with recall as a second check.

**Why this isn't just "train a model".**

- There is no ready-made truth label for "this title is bad". The hard part is defining a fair expected CTR (by position, and possibly by intent, content type and client), then checking that the gap is real and lasting rather than noise.
- The baseline is a plain rule: compare each page to the median CTR of its position tier. ML only earns its place if it predicts expected CTR better than that table, for example by treating position as continuous and accounting for intent and content type.
- In later weeks I plan to use the warehouse's daily data to test whether the gap persists: flag pages in an earlier window, then check the next window. That gives an observed outcome to measure against, instead of a gap I define and then "predict".

**Framing in one paragraph.** For an SEO editor deciding which pages to review first, I will build a ranked queue from search performance data, scoring each page by its position-adjusted CTR gap, measured by precision@K against an observed future outcome (gap persistence or later CTR change). A wrong call costs review time or missed clicks. A plain rule might not be enough because expected CTR shifts with exact position, intent and client, not just the tier bucket. I will claim only observed, directional, decision-support results.

In [2]:
# The candidate pool this lane works on
pool = df[has_pos & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].copy()
print(f"Candidate pages (position 1-20, >= 500 impressions/90d): {len(pool):,} "
      f"from {pool['client_id'].nunique()} clients")
print(f"Share of all impressions in the data that these pages hold: "
      f"{pool['impressions_90d'].sum() / df['impressions_90d'].sum():.1%}")

Candidate pages (position 1-20, >= 500 impressions/90d): 12,023 from 28 clients
Share of all impressions in the data that these pages hold: 76.1%


## 3. Quick look at the data (2-3 real numbers)

Reminder from the data dictionary: `ctr` is already a percentage, so 0.24 means 0.24%, not 24%.

**Number 1: pages at the same position get very different CTRs.** If CTR were mostly decided by position, pages in the same tier would sit close together. The cell below shows the spread inside each tier.

In [3]:
# CTR spread inside each position tier (pages with >= 500 impressions)
v = df[has_pos & (df["impressions_90d"] >= 500)]
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
spread = (v.groupby("position_tier")["ctr"]
            .agg(pages="size",
                 p25=lambda s: s.quantile(0.25),
                 median="median",
                 p75=lambda s: s.quantile(0.75))
            .reindex(order))
print(spread.round(2).to_string())
p1 = spread.loc["page_1"]
print(f"\npage_1: CTR at the 75th percentile is {p1['p75'] / p1['p25']:.1f}x the CTR at the 25th, "
      f"at roughly the same position")

               pages   p25  median   p75
position_tier                           
top_3            458  0.08    0.20  0.49
page_1          7064  0.12    0.24  0.44
striking        4485  0.08    0.17  0.34
page_3_5        4330  0.01    0.09  0.19
deep             389  0.00    0.00  0.04

page_1: CTR at the 75th percentile is 3.7x the CTR at the 25th, at roughly the same position


**Number 2: a lot of visible pages sit far below their tier.** Here a page is "under-capturing" if its CTR is under half the median of its own position tier.

In [4]:
pool["tier_median"] = pool.groupby("position_tier")["ctr"].transform("median")
under = pool["ctr"] < 0.5 * pool["tier_median"]
print(f"Under half their tier's median CTR: {under.sum():,} of {len(pool):,} pages ({under.mean():.0%})")
print(f"Their share of the pool's impressions: "
      f"{pool.loc[under, 'impressions_90d'].sum() / pool['impressions_90d'].sum():.0%}")
print(f"Pages with >= 500 impressions and zero clicks in 90 days: {(pool['clicks_90d'] == 0).sum():,}")

Under half their tier's median CTR: 3,031 of 12,023 pages (25%)
Their share of the pool's impressions: 18%
Pages with >= 500 impressions and zero clicks in 90 days: 1,215


**Number 3: the gap is concentrated, so a short queue covers much of it.** A rough size check: how many clicks would these pages get if each one only reached its tier's median CTR? This is not a forecast of what an edit would deliver. It only shows whether ranking pages by the gap puts the biggest gaps at the top of a list an editor could finish.

In [5]:
pool["click_gap"] = (pool["tier_median"] - pool["ctr"]).clip(lower=0) / 100 * pool["impressions_90d"]
total_gap = pool["click_gap"].sum()
ranked = pool.sort_values("click_gap", ascending=False)
print(f"Click gap to tier median, summed over the pool: {total_gap:,.0f} "
      f"(actual clicks in the pool: {pool['clicks_90d'].sum():,})")
for k in (20, 50, 200):
    print(f"Top {k:>3} pages hold {ranked['click_gap'].head(k).sum() / total_gap:.0%} of that gap")

Click gap to tier median, summed over the pool: 53,633 (actual clicks in the pool: 425,112)
Top  20 pages hold 12% of that gap
Top  50 pages hold 20% of that gap
Top 200 pages hold 41% of that gap


**What I take from these numbers (observed, starter slice only).**

- In the page_1 tier, a page at the 75th percentile gets about 3.7x the CTR of a page at the 25th percentile, at roughly the same position. So position alone leaves a lot unexplained, and that unexplained part is what this lane is about.
- About a quarter of visible candidate pages (3,031 of 12,023) are under half their tier median, holding 18% of the pool's impressions. 1,215 pages had at least 500 impressions and zero clicks.
- The top 50 pages by gap hold about 20% of the total gap. A reviewer working down a ranked list would reach a large share of the opportunity quickly, if the gaps turn out to be real.

## 4. Careful words: what I can and can't claim

**What this work can say:**

- Observed: in this data, page X got fewer clicks than most pages at a similar position, given its impressions.
- Directional: pages with certain traits (intent, content type, age) tend to sit further below their tier's typical CTR.
- Decision-support: these pages look most worth a human review first, and here is why (reason codes).
- If I validate on later windows in the warehouse: flagged pages kept (or did not keep) their gap in the next period.

**What it can't say:**

- That a title rewrite *caused* any later change in clicks. That needs an experiment, and this data has none.
- Anything about how Google ranks or displays results. I'm describing measured clicks and impressions, not the algorithm.
- That low CTR always means a bad title. It can also come from the query mix, branded vs non-branded searches, SERP features, or AI answers taking the click.
- That the click-gap number is clicks waiting to be won. It's a ranking device, not a promise.

**Known weak spots to handle later** (checked in the cell below):

- One client holds a large share of the candidate pages, so pooled numbers can mostly reflect that client. I'll report per-client results and use client-grouped splits.
- The top_3 tier shows a *lower* median CTR than page_1 (0.20% vs 0.24%), which is backwards, even with a 500-impression floor. It's also a small tier here (458 pages), and the data dictionary warns its CTR is unreliable in the starter slice. Query mix or SERP features could explain it, but I haven't checked. I won't trust top_3 tier medians on this slice and will recheck on the warehouse.
- The 90-day window mixes changes in position over time. A page's average position may hide days at position 2 and days at position 15.

In [6]:
# Checks behind the weak spots above
by_client = pool["client_id"].value_counts()
print(f"Largest single client's share of candidate pages: {by_client.iloc[0] / len(pool):.0%} "
      f"({by_client.iloc[0]:,} of {len(pool):,})")
print(f"Clients with fewer than 20 candidate pages: {(by_client < 20).sum()} of {len(by_client)}")

tier_check = (df[has_pos & (df["impressions_90d"] >= 500)]
                .groupby("position_tier")
                .agg(pages=("ctr", "size"), median_ctr=("ctr", "median"),
                     median_impressions=("impressions_90d", "median"))
                .loc[["top_3", "page_1"]])
print("\n" + tier_check.round(2).to_string())

Largest single client's share of candidate pages: 37% (4,395 of 12,023)
Clients with fewer than 20 candidate pages: 8 of 28

               pages  median_ctr  median_impressions
position_tier                                       
top_3            458        0.20              4263.0
page_1          7064        0.24              4504.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.